In [1]:
import psycopg2
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from plotnine import *
from sklearn.impute import SimpleImputer
from sqlalchemy import create_engine

In [2]:
# Load AWS Keys
load_dotenv()
user = os.environ.get('username')
pw = os.environ.get('password')

In [3]:
# Connecting to database
RDS_HOST     = 'endgbv-database.cx62ou6gkd87.us-east-2.rds.amazonaws.com'      
RDS_PORT     = 5432
RDS_DBNAME   = 'postgres'
RDS_USER     = user
RDS_PASSWORD = pw

conn = psycopg2.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    dbname=RDS_DBNAME,
    user=RDS_USER,
    password=RDS_PASSWORD
)

In [4]:
# Get Schema
schema = pd.read_sql(
    '''
    SELECT *
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_NAME = 'endgbv'
    ;
    ''', conn
)

schema

C:\Users\helen\AppData\Local\Temp\ipykernel_33596\4001415382.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


,table_catalog,table_schema,table_name,column_name,ordinal_position,column_default,is_nullable,data_type,character_maximum_length,character_octet_length,...,is_identity,identity_generation,identity_start,identity_increment,identity_maximum,identity_minimum,identity_cycle,is_generated,generation_expression,is_updatable
0,postgres,public,endgbv,Unemployment,15,None,YES,double precision,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
1,postgres,public,endgbv,Incident Precinct Code,3,None,YES,bigint,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
2,postgres,public,endgbv,Victim Reported Age,8,None,YES,double precision,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
3,postgres,public,endgbv,COMMDIST,12,None,YES,double precision,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
4,postgres,public,endgbv,Poverty,13,None,YES,double precision,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
5,postgres,public,endgbv,Median Income,14,None,YES,double precision,None,NaN,...,NO,None,None,None,None,None,NO,NEVER,None,YES
6,postgres,public,endgbv,Victim Sex,7,None,YES,text,None,1.073742e+09,...,NO,None,None,None,None,None,NO,NEVER,None,YES
7,postgres,public,endgbv,Offense Type,1,None,YES,text,None,1.073742e+09,...,NO,None,None,None,None,None,NO,NEVER,None,YES
8,postgres,public,endgbv,Suspect Race,9,None,YES,text,None,1.073742e+09,...,NO,None,None,None,None,None,NO,NEVER,None,YES
9,postgres,public,endgbv,Suspect Sex,10,None,YES,text,None,1.073742e+09,...,NO,None,None,None,None,None,NO,NEVER,None,YES


In [18]:
# Retrieve Data
df = pd.read_sql(
    '''
    SELECT
        "Offense Type", "Report Date", "Incident Precinct Code",
        "Borough Name", "Victim Reported Age", "Suspect Reported Age",
        "COMMDIST",
        CASE
            WHEN "Unemployment" = 1 THEN 'Yes'
            ELSE 'No'
        END AS "Unemployment",
        CASE
            WHEN "Median Income" = 1 THEN 'Yes'
            ELSE 'No'
        END AS "Median Income",
        CASE
            WHEN "Poverty" = 1 THEN 'Yes'
            ELSE 'No'
        END AS "Poverty",
        CASE 
            WHEN "Intimate Relationship Flag" = 'NO' THEN 'No'
            WHEN "Intimate Relationship Flag" = 'YES' THEN 'Yes'
            WHEN "Intimate Relationship Flag" = 'MISSING' THEN NULL
            WHEN "Intimate Relationship Flag" IS NULL THEN NULL
            ELSE "Intimate Relationship Flag"
        END AS "Intimate Relationship Flag",
        CASE
            WHEN "Victim Race" = 'None' THEN NULL
            WHEN "Victim Race" IS NULL THEN NULL
            ELSE "Victim Race"
        END AS "Victim Race",
        CASE 
            WHEN "Victim Sex" = 'None' THEN NULL
            WHEN "Victim Sex" IS NULL THEN NULL
            ELSE "Victim Sex"
        END AS "Victim Sex",
        CASE
            WHEN "Suspect Race" = 'None' THEN NULL
            WHEN "Suspect Race" IS NULL THEN NULL
            ELSE "Suspect Race"
        END AS "Suspect Race",
        CASE 
            WHEN "Suspect Sex" = 'None' THEN NULL
            WHEN "Suspect Sex" = 'UNKNWN' THEN NULL
            WHEN "Suspect Sex" IS NULL THEN NULL
            ELSE "Suspect Sex"
        END AS "Suspect Sex",
        CASE
            WHEN DATE("Report Date") > '2020-06-08' THEN 'Yes'
            ELSE 'No'
        END AS after_reopening,
        TO_CHAR(DATE("Report Date"), 'YYYY-MM') AS month_year
    FROM endgbv;
    ''', conn
)

C:\Users\helen\AppData\Local\Temp\ipykernel_33596\3072120136.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


In [20]:
# Recode NAs
df = df.replace({None: np.nan})

In [21]:
# Check Missingness
missingness_df = df.isna().mean().reset_index(name="pct_missing").assign(more_than_25 = lambda x: x["pct_missing"] > 0.25)
missingness_df


,index,pct_missing,more_than_25
0,Offense Type,0.000000,False
1,Report Date,0.000000,False
2,Incident Precinct Code,0.000000,False
3,Borough Name,0.000000,False
4,Victim Reported Age,0.287865,True
5,Suspect Reported Age,0.378238,True
6,COMMDIST,0.000215,False
7,Unemployment,0.000000,False
8,Median Income,0.000000,False
9,Poverty,0.000000,False


In [22]:
#remove cols where more than 25% of data is missing
df = df.drop(columns = missingness_df[missingness_df["more_than_25"] == True]["index"].tolist())

In [24]:
#convert everything to category except date
for col in df.columns:
    if col != "Report Date" and col != "month_year":
        df[col] = df[col].astype("category")
    else:
        df[col] = df[col].astype("object")

In [25]:
#create copy for imputation
impute_copy = df.copy()

In [26]:
#list of columns to impute
categorical_cols = [col for col in impute_copy.columns if col != "Report Date" and col != "month_year"]

In [27]:
#impute data
imputer = SimpleImputer(strategy="most_frequent")
imputed_array = imputer.fit_transform(impute_copy[categorical_cols])

In [28]:
#add in the imputed data
impute_copy[categorical_cols] = pd.DataFrame(imputed_array, columns = impute_copy[categorical_cols].columns)

In [29]:
#create copy for dataframe with missing data
raw_copy = df.copy()

In [30]:
#filling nas for nonimputed copy
for col in raw_copy.columns:
    fill_val = f"unknown_{col.replace(' ', '_')}".lower()
    if hasattr(raw_copy[col], 'cat'):
        raw_copy[col] = raw_copy[col].cat.add_categories(fill_val)
    raw_copy[col] = raw_copy[col].fillna(fill_val)

In [33]:
# Connect to RDS
engine = create_engine(
    'postgresql://' + user + ":" + pw + 
    "@endgbv-database.cx62ou6gkd87.us-east-2.rds.amazonaws.com" + 
    ":5432/postgres?sslmode=require"
)


In [ ]:
# Upload Imputed Dataset
impute_copy.to_sql('imputed_tbl', engine, if_exists='replace', index=False)

Done! Rows loaded: 483790


In [35]:
# Upload Non Imputed Dataset
raw_copy.to_sql('nonimputed_tbl', engine, if_exists='replace', index=False)

Done! Rows loaded: 483790
